# Similarity Search Benchmark

This notebook aims to provide a reproducible benchmark of given algorithms on similarity search. That is, given a dataset of Adaptive Immune Receptor sequences and a Levenshtein distance threshold, the algorithm identifies all pairs of sequences that have a similarity below the threshold value. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

The notebook is divided into 6 steps as follows:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires high-RAM VM which requires paid Google Colab account.
2. __Benchmark Setup:__ install dependencies and implementations of various algorithms and compile them if need be.
3. __Algorithm Benchmark:__ perform benchmark on 5 algorithms implemented in Python to measure the speed of each approach at threshold `d=1,2`. They include exhautive search, bk-tree, representation design, combinatorial lookup, and symmetric deletion.
4. __Symmetric Deletion Benchmark:__ perform benchmark on each variation of the implementation. They include Python implementation, XTNeighbor, and XTNeighbor without streaming.
5. __Result Download:__ download the benchmark measurement as csv file.

Warning: some sections take up to 1 hour to run. Indicative timings are provided for each step.

## 1. Configuration

In [ ]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = True # @param {type:"boolean"}

## 2. Benchmark Setup (run time < 5 min)

install dependency

In [ ]:
! pip install -q pyrepseq pybktree symscan

In [ ]:
import os.path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import random
import pybktree
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

import sys

import benchutils
from benchutils import BenchmarkTimeout, run_binary, time_limit, describe_env

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

In [ ]:
benchutils.timeout_seconds = 100

In [ ]:
describe_env()

compile XTNeighbor without streaming

In [ ]:
! mkdir -p {repo_path}/xtneighbor/build
! cd {repo_path}/xtneighbor/build; cmake -S .. -B .;make

compile XTNeighbor

In [ ]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

prepare input data

In [ ]:
N_FILES=6

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'{repo_path}/data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

In [ ]:
# garbage collect following large data import
# freeze imported data to avoid reconsideration during benchmarking
import gc
gc.collect()
gc.freeze()

## 2. Algorithm Benchmark (run time ~ 15 min at n_repeat=1)

exhaustive search implementation

In [ ]:
def exhaustive_search(seqs, max_edits):
  ans = []
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=max_edits)
        if dist <= max_edits:
            ans += [(i, j, dist)]
  return ans

bktree implementation

In [ ]:
def build_index(seqs):
    ans = {}
    for index, seq in enumerate(seqs):
        if seq not in ans:
            ans[seq] = []
        ans[seq].append(index)
    return ans


def bktree(seqs, max_edits=1):
    ans = []
    index = build_index(seqs)

    tree = pybktree.BKTree(levenshtein_distance, np.unique(seqs))
    for x_index, x_seq in enumerate(seqs):
        for edit_distance, y_seq in tree.find(x_seq, max_edits):
            for y_index in index[y_seq]:
                if x_index != y_index:
                    ans.append((x_index, y_index, edit_distance))
    return ans

benchmarking code

In [ ]:
# 100 is the warm up
sizes = [100, 1_000, 3_000, 10_000, 30_000, 100_000]
algorithms = {
    'kdtree':pyrepseq.kdtree,
    'symdel':pyrepseq.symdel,
    'combinatorial_lookup': pyrepseq.hash_based,
    'exhaustive_search': exhaustive_search,
    'bktree': bktree,
    'symscan': symscan.get_neighbors_within,
    }
limits = {}
#    'exhaustive_search_1':100_000,
#    'exhaustive_search_2':100_000,
#    'combinatorial_lookup_2':10_000,
#    'kdtree_2':100_000,
#    'exhaustive_search_2':100_000,
#    'bktree_2': 100_000}
limits.update({f"combinatorial_lookup_{dist}": 0 for dist in [4,5]})

alg_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_alg_exp(distances, sizes=sizes, verbose=False):
    for i in range(n_repeat):
        for distance in distances:
            for size in sizes:
                subset = random.Random(i).sample(data, size)
                for alg_name in algorithms:
                    limit = limits.get(f"{alg_name}_{distance}")
                    if limit is not None and limit < size:
                        if verbose:
                            print(f'Skipping {alg_name} with distance {distance} and size {size:,} because limit is {limit:,}')
                        continue

                    if verbose:
                        print(f'Running {alg_name} with distance {distance} and size {size:,}')

                    # perform
                    start = time.time()
                    try:
                        with time_limit():
                            algorithms[alg_name](subset, distance)
                    except BenchmarkTimeout:
                        print(f'Timeout: {alg_name} with distance {distance} and size {size:,} exceeded {benchutils.timeout_seconds}s')
                        limits[f"{alg_name}_{distance}"] = size - 1
                        continue
                    end = time.time()

                    # record
                    print(f'{size:,}', alg_name, distance, i, round((end-start)*100)/100)
                    alg_result['runtime'].append(end-start)
                    alg_result['algorithm'].append(alg_name)
                    alg_result['input_size'].append(size)
                    alg_result['distance'].append(distance)

In [ ]:
run_alg_exp(distances=[1])

In [ ]:
run_alg_exp(distances=[2])

In [ ]:
pd.DataFrame(alg_result).to_csv('../data/cpu_benchmark.csv')
if colab:
    files.download('cpu_benchmark.csv')

In [ ]:
alg_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

run_alg_exp(distances=range(1,6), sizes=[10_000], verbose=False)

In [ ]:
pd.DataFrame(alg_result).to_csv('../data/cpu_dist_benchmark.csv')
if colab:
    files.download('cpu_dist_benchmark.csv')

## 3. Symmetric Deletion  Implementation Benchmark (run time ~ 10 min at n_repeat=1)

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

symscan measurement worker (run out-of-process, see run_symscan below)

In [ ]:
! mkdir -p tmp

In [ ]:
%%writefile tmp/symscan_worker.py
# Standalone worker run as a fresh subprocess for each symscan measurement, so that a
# benchmark timeout can SIGKILL the whole process group instead of unwinding this
# interpreter. 
# Prints the seconds spent inside get_neighbors_within; interpreter startup and input
# reading are excluded, so the number is comparable to the in-process wall-clock
# timings of the other algorithms.
import sys
import time

import symscan


def main():
    seq_file, max_distance = sys.argv[1], int(sys.argv[2])

    with open(seq_file) as f:
        seqs = [line.strip() for line in f if line.strip()]

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()


standardize all algorithms to the same API

In [ ]:
def prepare(seqs):
  with open("input1.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)
  with open("input2.txt","w") as file2:
    file2.writelines(seq+'\n' for seq in (['cdr3']+seqs))

def xt_neighbor_1(seqs,threshold,_len,verbose=False): #verbose is ignored
  run_binary([f'{repo_path}/xtneighbor/build/xt_neighbor',
              '-p', 'input1.txt', '-n', str(_len), '-d', str(threshold)])

def xt_neighbor_2(seqs,threshold,_len,verbose=False):
  cmd = [f'{repo_path}/xtneighbor_streaming/build/xt_neighbor',
         '-i', 'input2.txt', '-n', str(_len), '-d', str(threshold)]
  if verbose:
    cmd.append('-V')
  run_binary(cmd)

def symdel(seqs,threshold,_len,verbose=False): #verbose is ignored
  return pyrepseq.symdel(seqs,max_edits=threshold)

SYMSCAN_WORKER = 'tmp/symscan_worker.py'

def run_symscan(seqs,threshold,_len,verbose=False):
  # measured inside the worker, so interpreter startup and reading input1.txt are
  # excluded; returns that runtime instead of letting the loop time the subprocess
  out = run_binary([sys.executable, SYMSCAN_WORKER, 'input1.txt', str(threshold)],
                   echo=verbose)
  try:
    return float(out.strip().splitlines()[-1])
  except (ValueError, IndexError):
    raise RuntimeError(f'{SYMSCAN_WORKER} produced no runtime:\n{out}') from None



benchmarking code

In [ ]:
# 100 is the warm up
sizes = [100, 10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000, 30_000_000]

limits = {}
#    'symscan_1':30_000_000,
#    'symscan_2':10_000_000,
#    'V1_1':10_000_000,
#    'V1_2':1_000_000,
#    'symdel_1':3_000_000,
#    'symdel_2':1_000_000,
#    'V2_1':30_000_000,
#    'V2_2':10_000_000}
limits.update({f"V1_{dist}": 0 for dist in [3,4,5]})

algorithms = {
    'V2':xt_neighbor_2,
    'symdel':symdel,
    'symscan':run_symscan,
}

# algorithms that measure themselves (out-of-process) and return their own runtime
# instead of being timed by the loop's wall clock
self_timed = {'symscan'}
verbose = False

symdel_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_exp(distances, sizes=sizes):
    for i in range(n_repeat):
        for distance in distances:
            for size in sizes:
                subset = random.Random(i).sample(data,size)
                prepare(subset)
                for alg_name in algorithms:
                    limit = limits.get(f"{alg_name}_{distance}")
                    if limit is not None and limit < size:
                        print(f"Skipping {alg_name} for size {size:,} and distance {distance} due to limit {limit}")
                        continue

                    # perform
                    start = time.time()
                    try:
                        with time_limit():
                            measured = algorithms[alg_name](subset,distance,size,verbose)
                    except BenchmarkTimeout:
                        print(f'Timeout: {alg_name} for size {size:,} and distance {distance} exceeded {benchutils.timeout_seconds}s')
                        limits[f"{alg_name}_{distance}"] = size - 1
                        continue
                    end = time.time()
                    runtime = measured if alg_name in self_timed else end - start

                    # record
                    print(f'{size:,}', alg_name, distance, i, round(runtime*100)/100)
                    symdel_result['runtime'].append(runtime)
                    symdel_result['algorithm'].append(alg_name)
                    symdel_result['input_size'].append(size)
                    symdel_result['distance'].append(distance)

In [ ]:
run_exp(distances=[1])

In [ ]:
run_exp(distances=[2])

In [ ]:
pd.DataFrame(symdel_result).to_csv('../data/gpu_benchmark.csv')
if colab:
    files.download('gpu_benchmark.csv')

In [ ]:
symdel_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

run_exp(distances=range(1,6), sizes=[100_000])

In [ ]:
pd.DataFrame(symdel_result).to_csv('../data/gpu_dist_benchmark.csv')
if colab:
    files.download('gpu_dist_benchmark.csv')